# Hito 2 — Análisis de coyuntura inicial: Tasa de Política Monetaria (TPM)
**Equipo:** Diego Alarcon, Francisco Vera, Martyn Sanhueza, Benjamin Tomckowiack · *Ciencia de Datos para Economía (UDP)*

> Cifras con corte a **octubre de 2026** (último dato diario: 19-oct-2026). Octubre está incompleto (solo TPM, expectativas y tasas de mercado), por lo que los comentarios usan el último dato disponible de cada serie.

In [1]:
import os, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

nombres = ["macro.db"]
carpetas = ["../db/", "db/", "../data/", "data/", ""]
DB = next((f"{c}{n}" for c in carpetas for n in nombres if os.path.exists(f"{c}{n}")), None)

ARCHIVOS_SQL = ["01_vistas.sql", "04_resumen_por_bloque.sql", "05_resumen_tpm_anual.sql", "06_huecos_mensuales.sql"]
def _hay_sql():
    return all(any(os.path.exists(f"{c}{a}") for c in ["../sql/", "sql/", ""]) for a in ARCHIVOS_SQL)

if DB is None or not _hay_sql():
    try:
        from google.colab import files
        if DB is None:
            print("Sube macro.db"); _ = files.upload()
            DB = next((n for n in nombres if os.path.exists(n)), None)
        if not _hay_sql():
            print("Sube los 4 archivos .sql de la carpeta sql/ del proyecto"); _ = files.upload()
    except ImportError:
        pass
assert DB, "No se encontro macro.db. Corre antes: python src/carga_db.py --hasta AAAA-MM-DD"
print("Usando base:", DB)

def leer_sql(nombre):
    """Lee <nombre> desde sql/ (repo) o desde la carpeta actual (Colab). La fuente unica de las consultas es el archivo .sql."""
    ruta = next((r for r in [f"../sql/{nombre}", f"sql/{nombre}", nombre] if os.path.exists(r)), None)
    assert ruta, f"No se encontro {nombre}: ejecuta el notebook desde notebooks/ (repo) o sube los .sql en Colab"
    return open(ruta, encoding="utf-8").read()

con = sqlite3.connect(DB)
con.executescript(leer_sql("01_vistas.sql"))

plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

panel = pd.read_sql("SELECT * FROM v_panel ORDER BY mes", con, parse_dates=["mes"]).set_index("mes")
d12   = pd.read_sql("SELECT * FROM v_panel_d12 ORDER BY mes", con, parse_dates=["mes"]).set_index("mes")
p = panel.loc["2015-01-01":]
print("Panel:", panel.index.min().date(), "->", panel.index.max().date(), "|", panel.shape[1], "columnas")

Usando base: ../db/macro.db
Panel: 2003-01-01 -> 2026-10-01 | 23 columnas


## 0. Calidad de los datos (consultas SQL con JOINs y GROUP BY)
Cobertura y datos faltantes por serie (`sql/04_resumen_por_bloque.sql`) y diagnóstico de huecos mensuales (`sql/06_huecos_mensuales.sql`).

In [2]:
display(pd.read_sql(leer_sql("04_resumen_por_bloque.sql"), con))
print("Meses sin dato dentro del rango de cada serie (solo aparecen las series con huecos):")
display(pd.read_sql(leer_sql("06_huecos_mensuales.sql"), con))

,bloque,alias,frecuencia,unidad,n_obs,n_faltantes,desde,ultimo_dato
0,Politica monetaria,tpm,Diaria,%,5933,2759,2003-01-02,2026-10-19
1,Expectativas,eee_12m,Mensual,%,286,0,2003-01-01,2026-10-01
2,Expectativas,eee_24m,Mensual,%,286,0,2003-01-01,2026-10-01
3,Precios,ipc_12m,Mensual,%,285,0,2003-01-01,2026-09-01
4,Precios,ipc_sv,Mensual,%,285,0,2003-01-01,2026-09-01
5,Actividad,imacec,Mensual,indice,284,0,2003-01-01,2026-08-01
6,Empleo,desempleo,Mensual,%,198,0,2010-03-01,2026-08-01
7,Tipo de cambio,dolar,Mensual,CLP/USD,285,0,2003-01-01,2026-09-01
8,Tipo de cambio,tcr,Mensual,indice,284,0,2003-01-01,2026-08-01
9,Tasas,bcp5,Diaria,%,5403,3285,2003-01-02,2026-10-15


Meses sin dato dentro del rango de cada serie (solo aparecen las series con huecos):


,alias,meses_sin_dato,primer_hueco,ultimo_hueco
0,bcp5,9,2022-10-01,2023-06-01
1,bcu5r,9,2022-10-01,2023-06-01
2,cpi_usa,1,2025-10-01,2025-10-01


**Interpretaciones**

Las 14 series cubren desde 2003 (el desempleo del INE parte en marzo de 2010). Los "**faltantes**" de las series diarias (TPM, BCP y BCU a 5 años) corresponden a fines de semana y feriados y no afectan el análisis, porque la vista mensual toma el último valor del mes. Las series con publicación más rezagada son Imacec, TCR y desempleo (agosto de 2026).

**Huecos conocidos (confirmados por la consulta 06).** El CPI de EE.UU. no tiene dato para octubre de 2025, y las tasas BCP y BCU a 5 años no tienen dato mensual entre octubre de 2022 y junio de 2023 (9 meses). En esos meses la API devuelve valores nulos para todos los días, por lo que no es un problema de carga. Ese hueco cae justo en el máximo de la TPM (11,25%) y afecta a la variable explicativa con mayor correlación (BCP a 5 años), así que antes del Hito 3 hay que decidir cómo tratarlo (serie alternativa, interpolación o restringir la muestra). Por ahora **no se interpola**: los gráficos muestran el corte y las correlaciones usan solo los pares con dato.

**Cómo se calculan las variaciones a 12 meses.** Con funciones de ventana en SQL (`PARTITION BY codigo`) ordenadas por un índice mensual entero y `RANGE BETWEEN 12 PRECEDING AND 12 PRECEDING`: toma exactamente el mes de hace 12 meses y devuelve NULL si ese mes no existe. Un `LAG(valor, 12)` simple se descuadra cuando falta un mes (ver `sql/01_vistas.sql`). El resultado coincide con el cálculo anterior (por JOIN de fechas) en todas las columnas del panel.

**Decisiones de datos y empalmes (a documentar en el informe):** (i) el IPC empalmado base 2018 terminaba en dic-2023, se reemplazó por el IPC empalmado por el BCCh base 2023; (ii) el IPCX1 se discontinuó en 2018 y se usa el IPC sin volátiles como medida subyacente; (iii) la tasa BCU a 5 años licitada terminó en 2017 y se reemplazó por tasas de mercado secundario BCP (nominal) y BCU/BTU (real) a 5 años; (iv) las expectativas de la EEE corresponden a horizontes de **11 y 23 meses**, no 12 y 24 (en la base conservan los alias históricos `eee_12m` y `eee_24m`; el panel las expone como `eee_11m` y `eee_23m`). El empalme del BCCh no coincide exactamente con las variaciones oficiales del INE (p. ej., abril de 2026: 3,95% en la base vs. 4,0% del INE).

## 1. Función de gráficos

In [ ]:
def grafico(cols, titulo, ylabel="%", hline=None, sec=None, sec_label=None, fuente="BCCh (BDE)", sombrear=None):
    """cols/sec: lista de (columna, etiqueta, estilo). sec = eje derecho.
    sombrear: lista de (inicio, fin, texto) para marcar periodos sin dato."""
    fig, ax = plt.subplots()
    for c, etq, est in cols:
        ax.plot(p.index, p[c], est, label=etq, lw=2.4 if c == "tpm" else 1.5)
    if hline is not None:
        ax.axhline(hline, color="red", lw=1, ls=":")
    for ini, fin, txt in (sombrear or []):
        ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fin), color="gray", alpha=.18)
        ax.text(pd.Timestamp(ini), ax.get_ylim()[0], " " + txt, fontsize=7, color="dimgray", va="bottom")
    ax.set_ylabel(ylabel); ax.set_title(titulo, loc="left", fontweight="bold")
    h, l = ax.get_legend_handles_labels()
    if sec:
        ax2 = ax.twinx()
        for c, etq, est in sec:
            ax2.plot(p.index, p[c], est, label=etq, lw=1.5, color="tab:orange")
        ax2.set_ylabel(sec_label); ax2.spines["right"].set_visible(True); ax2.grid(False)
        h2, l2 = ax2.get_legend_handles_labels(); h += h2; l += l2
    ax.legend(h, l, fontsize=8, frameon=False, ncol=3, loc="upper left")
    fig.text(0.01, 0.005, f"Fuente: {fuente}. Período: {p.index.min():%Y-%m} a {p.index.max():%Y-%m}. Cálculos propios.",
             fontsize=7, color="gray")
    plt.tight_layout(rect=[0, 0.03, 1, 1]); plt.show()

## 2. Política monetaria, inflación y expectativas

In [ ]:
grafico([("tpm","TPM","k-"), ("ipc_12m","IPC 12m","b-"), ("ipc_sv_12m","IPC sin volátiles 12m","c-"),
         ("eee_11m","Expectativas 11m","g--"), ("eee_23m","Expectativas 23m","m--")],
        "TPM, inflación y expectativas (meta 3%)", hline=3)

**Interpratacion**:  A octubre de 2026 la TPM está en 4,5%, sin cambios durante todo 2026. La inflación total (4,07% en septiembre) se ubica sobre la meta de 3%, pero la subyacente medida por el IPC sin volátiles (3,19%) está cerca de ella: la brecha entre ambas sugiere que el exceso de inflación proviene más de componentes volátiles que de presiones generalizadas de precios (la sección 2.b muestra de dónde salió ese exceso). Las expectativas a 23 meses están exactamente en 3,0% desde marzo de 2023 (44 meses seguidos), lo que indica **expectativas ancladas** a la meta; las de 11 meses subieron a 3,1%–3,4% en los últimos tres meses, reflejando la inflación corriente. Con expectativas largas ancladas, el Banco Central tiene menos necesidad de reaccionar con fuerza ante desvíos transitorios, lo que es coherente con una TPM estable (mecanismo 2 de la propuesta).

### 2.b Episodio de abril de 2026: shock de combustibles
La tabla muestra el quiebre del primer semestre de 2026 (datos del panel SQL).

In [ ]:
cols_ep = ["ipc_12m", "ipc_sv_12m", "eee_11m", "eee_23m", "tpm_real_exante"]
ep = panel.loc["2026-01":, cols_ep].copy()
ep["ipc_total_menos_sv"] = ep["ipc_12m"] - ep["ipc_sv_12m"]
display(ep.round(2))

**Interpretacion:** En abril la inflación a 12 meses salta de 2,83% a 3,95% y las expectativas a 11 meses de 3,0% a 4,0%, mientras la inflación sin volátiles casi no se mueve (3,41% → 3,44%). Por eso la brecha entre inflación total y subyacente pasa de negativa a positiva (+0,51 pp en abril, hasta +0,95 pp en junio): el shock está en los componentes volátiles. Según el INE, el IPC de abril subió 1,3% mensual, el mayor registro desde julio de 2022, con la gasolina subiendo 25,3% en el mes y el diésel 45,7%, en un contexto de mayores precios internacionales del petróleo y del cambio en el mecanismo de estabilización de precios de combustibles (Mepco). *Fuentes: INE vía prensa, ver lista al final.*

Consecuencia para la TPM: como las expectativas a 11 meses subieron más que la TPM, la **TPM real ex ante** cayó de 1,5 a 0,5 pp en abril y se recuperó después (1,1 pp en octubre). El Consejo no reaccionó con alzas, coherente con tratar el shock como transitorio mientras las expectativas a 23 meses siguieron ancladas. Para el modelo del Hito 3 este episodio importa por dos razones: es un shock de oferta (no se explica con brecha de actividad) y puede generar valores atípicos en la regla de Taylor.

## 3. Postura monetaria: TPM real

In [ ]:
grafico([("tpm_real_exante","TPM real ex ante (TPM - exp. 11m)","k-"),
         ("tpm_real_expost","TPM real ex post (TPM - IPC 12m)","b-")],
        "Tasa de política monetaria real", ylabel="puntos porcentuales", hline=0)

In [ ]:
# TPM nominal frente al rango de tasa neutral nominal que el Banco Central publicó en el IPoM de dic-2025 (3,75%-4,75%)
q = panel.loc["2023-01-01":]
fig, ax = plt.subplots()
ax.plot(q.index, q["tpm"], "k-", lw=2.4, label="TPM")
ax.fill_between([pd.Timestamp("2025-12-01"), q.index.max()], 3.75, 4.75, color="tab:green", alpha=.15, label="Rango TPM neutral (IPoM dic-2025, vigente desde esa fecha)")
ax.set_ylabel("%"); ax.set_title("TPM y rango de tasa neutral nominal", loc="left", fontweight="bold")
ax.legend(fontsize=8, frameon=False, loc="upper right")
fig.text(0.01, 0.005, f"Fuente: BCCh (BDE) y BCCh, IPoM dic-2025 (rango neutral, vía Ex-Ante). Período: {q.index.min():%Y-%m} a {q.index.max():%Y-%m}.",
         fontsize=7, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1]); plt.show()

**Interpretacion:** La TPM real *ex ante* (TPM menos expectativas a 11 meses) fue de 1,1 a 1,5 puntos entre julio y octubre, mientras que la *ex post* (descontando la inflación observada) es bastante menor (0,43 en septiembre) porque la inflación efectiva supera a la esperada. Para juzgar si la política es restrictiva hay que compararla con la tasa neutra: en su IPoM de diciembre de 2025 el Banco Central estimó el rango de la TPM neutral nominal en 3,75%–4,75%. Con la TPM en 4,5%, la política está **dentro del rango neutral, en su mitad superior**: es, como mucho, levemente contractiva, y no una postura claramente restrictiva. Esto es consistente con que el mercado y el propio Banco Central hablen de un corredor de TPM cercano a la neutralidad. *Pendiente:* verificar si los IPoM de marzo, junio y septiembre de 2026 modificaron ese rango y citar la versión vigente en el informe final.

### Resumen anual de la TPM (`sql/05_resumen_tpm_anual.sql`: GROUP BY sobre el panel)

In [ ]:
display(pd.read_sql(leer_sql("05_resumen_tpm_anual.sql"), con).tail(12))

## 4. Actividad y mercado laboral

In [ ]:
grafico([("imacec_12m","Imacec var. 12m (%)","b-"), ("imacec_12m_mm3","Imacec var. 12m, media móvil 3m (%)","k-")],
        "Actividad y mercado laboral", hline=0,
        sec=[("desempleo","Desempleo (%, eje der.)","-")], sec_label="Desempleo %")

**Interpretacion:** El Imacec muestra variaciones anuales muy volátiles en los últimos meses (+2,02% en junio, -1,49% en julio y -1,01% en agosto). Parte de esa volatilidad es mecánica: la serie utilizada es la *serie original*, sin corrección por días hábiles ni desestacionalización, por lo que diferencias de calendario distorsionan la variación anual. Para no sobrerreaccionar a un mes, el semáforo y el gráfico usan la **media móvil de 3 meses** de la variación anual (calculada en SQL, `var_12m_mm3`), que a agosto es de -0,16%: una actividad prácticamente estancada, con una tendencia negativa suave desde comienzos de año. *Pendiente para el Hito 3:* reemplazar la serie por la versión desestacionalizada y corregida por calendario que publica el BCCh.

El desempleo (9,6% en agosto de 2026) está muy sobre su promedio de 2010–2019 (6,9%), lo que sugiere holgura en el mercado laboral. Esa lectura requiere cautela por dos razones: (i) el panel no incluye ninguna serie de salarios, así que no podemos afirmar nada sobre presiones salariales (inferirlas desde el desempleo es solo una hipótesis); (ii) según un resumen de Santander del IPoM de septiembre, el Banco Central atribuye el alto desempleo a una combinación de deterioro cíclico y de un mayor nivel neutral estructural, de modo que no todo el exceso sobre el promedio histórico es holgura cíclica. Agregar un índice de remuneraciones queda como mejora para el Hito 3.

*Limitación:* la brecha de actividad del panel (Imacec anual menos su promedio histórico expandido) da valores de -4 a -5 puntos, pero refleja principalmente que el crecimiento tendencial actual es menor al de los años 2000, no necesariamente una brecha del producto de esa magnitud. No se usa en este notebook y se reemplazará por una estimación tipo filtro HP calculada solo con información pasada.

## 5. Tipo de cambio y cobre

In [ ]:
grafico([("dolar","Dólar observado (CLP)","k-")], "Tipo de cambio y precio del cobre",
        ylabel="CLP/USD", sec=[("cobre","Cobre (USD/lb, eje der.)","-")], sec_label="USD/lb")

**Interpretacion:** El dólar promedió 947 pesos en septiembre, una depreciación del peso de cerca de 3% respecto de agosto (918), mientras el cobre se mantuvo entre 6,1 y 6,6 USD/lb de mayo a septiembre. Un peso más débil encarece los bienes transables importados y es un canal de traspaso hacia la inflación total (mecanismo 3 de la propuesta); el precio del cobre, por su parte, sostiene los términos de intercambio y el ingreso externo. Que el peso se deprecie aun con cobre alto indica que el movimiento de septiembre no se explica por el precio del cobre. Un posible factor es el externo (la Fed subió en septiembre, ver sección 6), pero **es solo una hipótesis**: el panel no incluye un índice del dólar global (p. ej., el índice amplio del dólar de FRED) que permita separar un movimiento global del dólar de uno específico de Chile.

## 6. Contexto externo: Chile vs EE.UU.

In [ ]:
grafico([("tpm","TPM","k-"), ("fedfunds","Fed Funds","r-"), ("diferencial_fed","TPM - Fed","g--")],
        "Política monetaria: Chile vs EE.UU.", hline=0, fuente="BCCh (BDE) y FRED")

**Interpretacion:** La tasa efectiva de la Fed subió de 3,63% en agosto a 3,75% en septiembre, y la inflación anual de EE.UU. (CPI) se aceleró de 3,3% en julio a 3,7% en septiembre. El diferencial TPM–Fed se estrechó a 0,75 puntos (desde 0,87). Un diferencial de tasas menor presiona al peso y reduce el margen del Banco Central para recortar sin generar presión cambiaria: es un **riesgo al alza para la trayectoria de la TPM** (o, al menos, un freno a eventuales recortes). Este es el tipo de condicionamiento externo del mecanismo 4 de la propuesta.

## 7. Tasas de largo plazo y expectativas de mercado

In [ ]:
grafico([("tpm","TPM","k-"), ("bcp5","BCP 5 años (nominal)","b-"), ("bcu5r","BCU/BTU 5 años (real)","c-")],
        "TPM y tasas de largo plazo", hline=0,
        sombrear=[("2022-10-01", "2023-06-01", "sin dato BCP/BCU")])

**Interpretacion:** La tasa BCP a 5 años (5,70% a mediados de octubre) está 1,2 puntos sobre la TPM. Esa pendiente era de 0,60 puntos en junio, alcanzó 1,43 en septiembre y se mantiene alta. Una curva con pendiente positiva y creciente indica que el mercado **no descuenta recortes de TPM** en el horizonte relevante, y puede reflejar expectativas de alzas o mayor prima por plazo (con estas series no se puede separar una de otra). Este hallazgo es importante para la proyección: sugiere que una trayectoria a la baja de la TPM enfrentaría evidencia de mercado en contra.La tasa real a 5 años (2,31%) es la referencia de largo plazo para el costo del crédito en UF.

## 8. Semáforo de coyuntura
Último dato disponible de cada indicador. **Los umbrales son supuestos del equipo** (a justificar en el informe); el criterio de cada fila se muestra en la tabla.

In [ ]:
def ultimo(col):
    s = panel[col].dropna()
    return s.index[-1], s.iloc[-1]

prom_des = panel.loc["2010":"2019", "desempleo"].mean()

V, A, R = "🟢", "🟡", "🔴"
reglas = [
 ("Inflación total (12m, %)",    "ipc_12m",         lambda v: V if 2<=v<=4 else (A if 1.5<=v<=4.5 else R), "Verde 2–4% | Amarillo 1,5–4,5%"),
 ("Inflación sin volátiles (%)", "ipc_sv_12m",      lambda v: V if 2<=v<=4 else (A if 1.5<=v<=4.5 else R), "Verde 2–4% | Amarillo 1,5–4,5%"),
 ("Expectativas 11m (%)",        "eee_11m",         lambda v: V if 2.5<=v<=3.5 else (A if 2<=v<=4 else R), "Verde 2,5–3,5% | Amarillo 2–4%"),
 ("Expectativas 23m (%)",        "eee_23m",         lambda v: V if 2.5<=v<=3.5 else (A if 2<=v<=4 else R), "Verde 2,5–3,5% | Amarillo 2–4%"),
 ("TPM real ex ante (pp)",       "tpm_real_exante", lambda v: V if 0.5<=v<=2 else (A if 0<=v<=3 else R),   "Verde 0,5–2 | Amarillo 0–3"),
 ("Imacec (var. 12m, media móvil 3m, %)", "imacec_12m_mm3", lambda v: V if v>=2 else (A if v>=0 else R),   "Verde ≥2 | Amarillo 0–2"),
 ("Desempleo (%)",               "desempleo",       lambda v: V if v<=prom_des+0.5 else (A if v<=prom_des+1.5 else R), f"Ref. 2010-19: {prom_des:.1f}%"),
 ("Dólar (var. 12m, %)",         "dolar_12m",       lambda v: V if abs(v)<5 else (A if abs(v)<10 else R),  "Verde |var|<5% | Amarillo <10%"),
]
filas = []
for nombre, col, f, crit in reglas:
    fecha, v = ultimo(col)
    filas.append({"Indicador": nombre, "Último dato": fecha.strftime("%Y-%m"), "Valor": round(v, 2),
                  "Semáforo": f(v), "Criterio": crit})
semaforo = pd.DataFrame(filas)
display(semaforo)

**Interpretacion:** El semáforo resume una coyuntura de **inflación subyacente cerca de la meta, inflación total algo sobre ella por el shock de combustibles, actividad estancada y mercado laboral holgado**: los indicadores de precios y expectativas están en verde o amarillo, mientras que actividad y empleo muestran las señales más débiles. La actividad aparece en rojo por poco (media móvil de 3 meses de -0,16%), así que conviene leerla como "estancamiento" y no como contracción marcada. La combinación de inflación contenida en lo subyacente, expectativas largas ancladas y actividad débil es coherente con una TPM estable en el corto plazo, aunque la pendiente de las tasas de mercado (sección 7) apunta en otra dirección. Este es el escenario base desde el cual se proyectará la trayectoria a 24 meses.

## 9. Relaciones y rezagos con la TPM
Correlación entre la TPM en *t* y cada variable en *t − l* (desde 2010). Se muestran los niveles, las variaciones a 12 meses (para controlar la persistencia; calculadas en SQL, vista `v_panel_d12`) y una prueba de sensibilidad que excluye el ciclo 2020–2023.

In [ ]:
cols = ["tpm","ipc_12m","ipc_sv_12m","eee_11m","eee_23m","imacec_12m","desempleo","fedfunds","dolar","cobre","bcp5"]
niv = panel.loc["2010":, cols]
var = d12.loc["2010":, cols]
rez = [0, 3, 6, 9, 12]

def tabla_corr(df, excluir=None):
    """Correlacion de la TPM en t con x en t-l. 'excluir' = (inicio, fin): se descartan los pares
    en que t o t-l caen en ese periodo."""
    out = {}
    for x in df.columns.drop("tpm"):
        fila = []
        for l in rez:
            xs, ys = df[x].shift(l), df["tpm"]
            if excluir:
                a, b = map(pd.Timestamp, excluir)
                fechas_l = pd.Series(df.index, index=df.index).shift(l)
                malo = ((df.index >= a) & (df.index <= b)) | ((fechas_l >= a) & (fechas_l <= b)).values
                xs, ys = xs.where(~malo), ys.where(~malo)
            fila.append(ys.corr(xs))
        out[x] = fila
    return pd.DataFrame(out, index=[f"x rezagada {l}m" for l in rez]).T.round(2)

print("A) Niveles");                                  display(tabla_corr(niv))
print("B) Variaciones a 12 meses (SQL)");             display(tabla_corr(var))
print("C) Sensibilidad: niveles, excluyendo 2020-2023"); display(tabla_corr(niv, ("2020-01-01", "2023-12-01")))
print("D) Sensibilidad: variaciones a 12 meses, excluyendo 2020-2023"); display(tabla_corr(var, ("2020-01-01", "2023-12-01")))

**Interpretaciones (niveles A y variaciones B):** (1) La TPM se asocia más con la inflación y las expectativas rezagadas 3 a 6 meses que con las contemporáneas (IPC: 0,81 en niveles a 6 meses y 0,75 en variaciones a 3 meses; expectativas a 11 meses: 0,85 y 0,81 a 6 meses). El Banco Central responde con rezago a la inflación y a lo que esperan los agentes, y la relación **se mantiene al controlar por persistencia**. (2) Las tasas de mercado anticipan a la TPM: la BCP a 5 años tiene correlación 0,82 a 3 meses en niveles y **0,89 a 6 meses en variaciones**, la más alta de ambas tablas. (3) El desempleo casi no se relaciona con la TPM en niveles (|r| ≤ 0,13), pero sí en variaciones: -0,54 a 6 meses, es decir, aumentos del desempleo anteceden recortes (o menores alzas) de la TPM, coherente con el canal de holgura laboral. (4) La Fed Funds tiene correlación contemporánea de 0,57 en niveles (0,48 en variaciones) que se diluye y se vuelve negativa con rezagos largos (-0,41 a 12 meses): ambas responden a un ciclo global común más que una liderar a la otra. El dólar, con 0,33 en niveles, casi no se asocia en variaciones (|r| ≤ 0,17).

*Nota técnica:* las variaciones a 12 meses ahora se calculan en SQL con toda la historia del panel, por lo que algunas correlaciones difieren en centésimas de las de la versión anterior (que calculaba las diferencias en pandas después de recortar la muestra en 2010).

**Sensibilidad (C y D, excluyendo 2020–2023).** Esta prueba cambia varias conclusiones y conviene reportarla:
- **Se mantienen:** la BCP a 5 años sigue siendo la variable más asociada (0,80 en niveles y 0,61 en variaciones a 6 meses) y las expectativas a 11 meses también (0,50 y 0,67). El signo negativo del desempleo en variaciones se conserva (-0,30 a 6 meses), aunque en niveles cambia de signo, lo que confirma que los niveles no son interpretables.
- **Se debilitan:** la correlación de la TPM con el IPC en niveles cae de 0,81 a 0,28 a 6 meses (en variaciones baja de 0,71 a 0,54). Buena parte de la asociación con la inflación viene del episodio 2021–2023, cuando la inflación y la TPM subieron juntas, y no de una regla estable.
- **Son frágiles:** la Fed Funds en variaciones queda en torno a cero y el dólar cambia de signo (en D es negativo, -0,50 a 3 meses). No se interpretan.
- **No se explican por 2020–2023:** el Imacec (0,63 a 12 meses en variaciones) y el cobre (0,51) siguen mostrando correlaciones que crecen con el rezago. Eso contradice la explicación inicial de que provenían solo del ciclo 2020–2022. Es más probable que reflejen tendencias compartidas en la muestra que un mecanismo directo; hay que tratarlas con pruebas de estacionariedad antes de usarlas como predictores.





**Fuentes externas citadas en los comentarios:**
- INE, IPC de abril de 2026 (+1,3% mensual, 4,0% anual; gasolina +25,3%, diésel +45,7%), vía El Desconcierto: https://eldesconcierto.cl/actualidad/ipc-abril-llega-al-13-el-mas-alto-4-anos-combustibles-y-transportes-golpean-el-bolsillo-los-chilenos-n5458647/amp
- Banco Central de Chile, IPoM diciembre 2025 (rango de TPM neutral 3,75%–4,75%), vía Ex-Ante: https://www.ex-ante.cl/?p=332198
- Santander, resumen del IPoM de septiembre (desempleo cíclico y estructural): https://banco.santander.cl/estudios/flash/detalles/ipom-de-septiembre-peor-hoy-mejor-manana